<a href="https://colab.research.google.com/github/AngelTroncoso/Feature-Engineering/blob/main/actividad7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Métodos de Filtro y Correlación: Limpiando el Ruido en Feature Engineering¡Hola! Estoy muy emocionado de que sigas aquí porque hoy vamos a aprender a limpiar el ruido de tus proyectos de datos. Imagina que cada variable es una voz en una habitación; si todos dicen lo mismo o alguien se queda callado, no hay una conversación real. Eso es justo lo que arreglaremos hoy con los **Métodos de Filtro**.### Objetivo de la PrácticaAprenderás a utilizar pruebas de varianza y matrices de correlación en Python para identificar y descartar variables redundantes o constantes que no aportan valor predictivo, optimizando así el rendimiento y la interpretabilidad de tus modelos de Ciencia de Datos.### ¿Por qué esto es importante?A menudo, los modelos de Machine Learning se vuelven lentos o imprecisos porque les damos *basura*. Si una columna tiene siempre el mismo valor (como el número de piernas en un estudio sobre humanos adultos), esa variable no discrimina ni enseña nada al modelo. Del mismo modo, si tenemos dos variables que dicen lo mismo (como la temperatura en Celsius y Fahrenheit), generamos redundancia o **multicolinealidad**, lo cual confunde al algoritmo.En esta sesión, aprenderás a ser un *cirujano de datos*, cortando lo que sobra para que lo verdaderamente importante brille.

## 1. Filtro de Varianza (VarianceThreshold)Comenzaremos con el concepto de varianza nula. Si un dato no cambia, no explica la diferencia entre un resultado y otro. Usaremos la herramienta `VarianceThreshold` de Scikit-learn para detectar variables con varianza cero o cercana a cero.### Escenario: Inventario IndustrialImagina que tenemos un dataset de 10,000 filas de un inventario donde existe una columna llamada `impuesto_fijo` que siempre vale 16. Vamos a simular este caso y eliminar esa columna automáticamente.

In [ ]:
import pandas as pdimport numpy as npfrom sklearn.feature_selection import VarianceThreshold# 1. Simulación de datos de inventarionp.random.seed(42)n_filas = 1000data_inventario = {    'id_producto': range(n_filas),    'unidades_stock': np.random.randint(0, 100, n_filas),    'precio_unitario': np.random.uniform(10.5, 500.0, n_filas),    'impuesto_fijo': [16.0] * n_filas,  # Esta es la columna constante    'costo_envio': np.random.normal(15, 2, n_filas)}df_inv = pd.DataFrame(data_inventario)print('Dimensiones originales:', df_inv.shape)print('Primeras filas del dataset:')print(df_inv.head())print()# 2. Aplicar VarianceThreshold# El umbral 0 elimina columnas donde todos los valores son igualesselector = VarianceThreshold(threshold=0)selector.fit(df_inv)# Obtener las columnas que se mantienencolumnas_seleccionadas = df_inv.columns[selector.get_support()]df_filtrado = df_inv[columnas_seleccionadas]print('Dimensiones después de VarianceThreshold:', df_filtrado.shape)print('Columnas eliminadas:', set(df_inv.columns) - set(df_filtrado.columns))

## 2. Matriz de Correlación y RedundanciaAhora que quitamos el silencio, hablemos de la redundancia. En estadística, cuando dos variables están íntimamente ligadas, se dice que tienen una alta correlación. Mantener ambas puede causar inestabilidad en modelos lineales.### El Caso de Carlos: Analista de RiesgosCarlos intentaba predecir fraudes bancarios con 200 variables. Su modelo era pesado y fallaba. Aplicó una matriz de correlación y descubrió que variables como **ingreso anual** e **impuesto pagado** decían casi lo mismo. Al limpiar su dataset, su modelo fue un 40% más rápido.### Escenario: Ventas InmobiliariasVamos a ver cómo se relacionan los `metros_cuadrados` y el `numero_de_habitaciones` en un dataset de casas.

In [ ]:
import seaborn as snsimport matplotlib.pyplot as plt# 1. Generar datos inmobiliarios con alta correlaciónn_casas = 100metros_cuadrados = np.random.normal(120, 40, n_casas)# El número de habitaciones depende fuertemente de los metros cuadradosnum_habitaciones = (metros_cuadrados / 30).astype(int) + np.random.randint(0, 2, n_casas)precio = (metros_cuadrados * 2000) + (num_habitaciones * 5000) + np.random.normal(0, 5000, n_casas)df_casas = pd.DataFrame({    'metros_cuadrados': metros_cuadrados,    'num_habitaciones': num_habitaciones,    'precio_venta': precio,    'antiguedad_anos': np.random.randint(0, 50, n_casas)})# 2. Calcular la matriz de correlación de Pearsonmatriz_corr = df_casas.corr()print('Matriz de Correlación:')print(matriz_corr)print()# 3. Visualización con Heatmapplt.figure(figsize=(8, 6))sns.heatmap(matriz_corr, annot=True, cmap='coolwarm', fmt='.2f')plt.title('Mapa de Calor de Correlaciones Inmobiliarias')plt.show()

## 3. Script para Eliminar Variables Altamente CorrelacionadasNo siempre podemos mirar gráficas si tenemos cientos de variables. Necesitamos un proceso automatizado. Imagina un negocio de logística donde `fecha_de_envio` y `dia_de_llegada` están tan ligadas que una de ellas sobra.A continuación, crearemos un script para identificar y eliminar variables que superen un umbral de correlación de 0.90.

In [ ]:
# 1. Simular datos de logística con variables gemelasdata_logistica = {    'distancia_km': np.random.uniform(10, 1000, 100),    'peso_kg': np.random.uniform(1, 50, 100),    'tiempo_estimado_horas': np.random.uniform(1, 24, 100)}df_log = pd.DataFrame(data_logistica)# Creamos una variable 'gemela' con ruido mínimodf_log['tiempo_estimado_minutos'] = df_log['tiempo_estimado_horas'] * 60 + np.random.normal(0, 0.1, 100)print('Columnas antes del filtro:', df_log.columns.tolist())# 2. Algoritmo para identificar redundanciaumbral = 0.90corr_matrix = df_log.corr().abs()# Seleccionar el triángulo superior de la matriz de correlaciónsuperior = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))# Encontrar índices de columnas con correlación mayor al umbralcolumnas_a_eliminar = [columna for columna in superior.columns if any(superior[columna] > umbral)]print('Variables candidatas a eliminación por alta correlación:', columnas_a_eliminar)# 3. Eliminar las variablesdf_log_final = df_log.drop(columns=columnas_a_eliminar)print('Columnas finales:', df_log_final.columns.tolist())print('¡Redundancia eliminada con éxito!')

## Reflexión Final y Profundización¿Por qué preferimos los métodos de filtro al inicio del flujo? A diferencia de otros métodos, los filtros miran las propiedades **intrínsecas** de los datos de forma aislada. Son extremadamente rápidos porque no necesitan entrenar un modelo entero.Sin embargo, ten cuidado:*   **No seas demasiado agresivo:** Si eliminas variables que cambian muy poquito (umbral de varianza muy alto), podrías perder señales sutiles.*   **Pearson tiene límites:** Solo detecta relaciones lineales. Si una variable crece en forma de curva, Pearson podría decir que no hay relación. ¡Siempre mira tus gráficos de dispersión!*   **Contexto de negocio:** Nunca borres algo sin entender qué es. A veces, esa pequeña diferencia entre dos variables gemelas es donde se esconde la oportunidad de predicción.¡Qué gran avance has hecho hoy! Dominar la limpieza de variables redundantes es lo que separa a un principiante de un verdadero profesional de la Ciencia de Datos. Recuerda: **menos es más** cuando hablamos de variables. ¡Sigue practicando y nos vemos en la siguiente lección!